In [0]:
%sql

USE CATALOG rocket;

CREATE SCHEMA IF NOT EXISTS gold;

CREATE TABLE IF NOT EXISTS gold.v_cliente_360;

CREATE TABLE IF NOT EXISTS gold.desempenho_produtos;

CREATE TABLE IF NOT EXISTS gold.analise_tickets;

CREATE TABLE IF NOT EXISTS gold.kpi_por_categoria;

CREATE TABLE IF NOT EXISTS gold.kpi_por_estado;

CREATE TABLE IF NOT EXISTS gold.kpi_por_status;

CREATE TABLE IF NOT EXISTS gold.comportamento_digital;

CREATE TABLE IF NOT EXISTS gold.pedidos_por_cliente;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import year, month, col, sum, round, avg, count, countDistinct

In [0]:
dim_clientes      = spark.read.table("rocket.silver.dim_clientes")
fat_pedidos       = spark.read.table("rocket.silver.fat_pedidos")
fat_avaliacoes    = spark.read.table("rocket.silver.fat_avaliacoes")
fat_tickets       = spark.read.table("rocket.silver.fat_suporte_tickets")
dim_catalogo      = spark.read.table("rocket.silver.dim_catalogo_produtos")
dim_clickstream   = spark.read.table("rocket.silver.dim_clickstream")

# V_cliente_360


In [0]:
# Métricas de pedidos
#Calcula, por cliente: total de compras realizadas, receita total gerada, ticket médio, data da primeira e da última compra. Identifica também o método de pagamento preferido, a categoria de produto mais comprada, e o produto mais comprado individualmente ambos via window function de ranking.

# Método de pagamento preferido
window_pagamento = Window.partitionBy("id_cliente").orderBy(F.desc("contagem"))

metodo_preferido = (
    fat_pedidos
    .groupBy("id_cliente", "metodo_pagamento")
    .agg(F.count("*").alias("contagem"))
    .withColumn("rank", F.row_number().over(window_pagamento))
    .filter(F.col("rank") == 1)
    .select("id_cliente", F.col("metodo_pagamento").alias("metodo_pagamento_preferido"))
)

# Categoria preferida 
pedidos_com_categoria = fat_pedidos.join(
    dim_catalogo.select("id_produto", "categoria"),
    "id_produto",
    "left"
)

window_categoria = Window.partitionBy("id_cliente").orderBy(F.desc("contagem_categoria"))

categoria_preferida = (
    pedidos_com_categoria
    .groupBy("id_cliente", "categoria")
    .agg(F.count("*").alias("contagem_categoria"))
    .withColumn("rank", F.row_number().over(window_categoria))
    .filter(F.col("rank") == 1)
    .select("id_cliente", F.col("categoria").alias("categoria_preferida"))
)

window_produto = Window.partitionBy("id_cliente").orderBy(F.desc("contagem_produto"))

produto_mais_comprado = (
    fat_pedidos
    .groupBy("id_cliente", "id_produto")
    .agg(F.count("*").alias("contagem_produto"))
    .withColumn("rank", F.row_number().over(window_produto))
    .filter(F.col("rank") == 1)
    .join(dim_catalogo.select("id_produto", "nome_produto"), "id_produto", "left")
    .select("id_cliente", F.col("nome_produto").alias("produto_mais_comprado"))
)

# Métricas gerais de pedidos
metricas_pedidos = (
    fat_pedidos
    .groupBy("id_cliente")
    .agg(
        F.count("id_pedido").alias("total_compras"),
        F.sum("valor_pedido").cast("decimal(15,2)").alias("receita_total_cliente"),
        F.round(F.avg("valor_pedido"), 2).cast("decimal(15,2)").alias("ticket_medio"),
        F.min("data_pedido").alias("data_primeira_compra"),
        F.max("data_pedido").alias("data_ultima_compra"),
    )
)

In [0]:
# Métricas de avaliações
#Conta quantas avaliações o cliente realizou e calcula a média da nota do produto e do NPS. Clientes sem avaliações aparecem com nulo nessas colunas.

metricas_avaliacoes = (
    fat_avaliacoes
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_produto"), 2).alias("media_nota_produto"),
        F.round(F.avg("nota_nps"), 2).alias("media_nota_nps"),
    )
)

In [0]:
# Métricas de tickets
#Conta o total de tickets abertos pelo cliente, separando entre tickets ainda abertos e tickets já resolvidos. Útil para o vendedor entender o histórico de problemas antes de uma abordagem.

metricas_tickets = (
    fat_tickets
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("total_tickets"),
        F.sum(F.when(F.col("status_ticket") == "aberto", 1).otherwise(0)).alias("tickets_abertos"),
        F.sum(F.when(F.col("status_ticket") == "fechado", 1).otherwise(0)).alias("tickets_fechados"),
    )
)

In [0]:
# Métricas de clickstream 
# Agrega dados de navegação do cliente: total de sessões, total de produtos visitados e tempo médio de sessão. Considera apenas eventos de clientes identificados, filtrando sessões anônimas.

metricas_click = (
    dim_clickstream
    .filter(F.col("flag_anonimo") == False)
    .groupBy("id_cliente")
    .agg(
        F.countDistinct("id_sessao").alias("total_sessoes"),
        F.count("id_produto").alias("total_produtos_visitados"),
        F.round(F.avg("tempo_pagina_seg"), 2).alias("tempo_medio_sessao_seg"),
    )
)

In [0]:
# Montagem final Adiciona a coluna segmento_cliente derivada de total_compras, receita_total_cliente e data_ultima_compra:
#- Novo: apenas 1 compra
#- Inativo: última compra há mais de 180 dias
#- Premium: mais de 5 compras ou receita acima de R$ 1.000
#- Recorrente: demais casos

v_cliente_360 = (
    dim_clientes
    .join(metricas_pedidos,    "id_cliente", "left")
    .join(metodo_preferido,    "id_cliente", "left")
    .join(categoria_preferida, "id_cliente", "left")
    .join(metricas_avaliacoes, "id_cliente", "left")
    .join(metricas_tickets,    "id_cliente", "left")
    .join(metricas_click,      "id_cliente", "left")
    .join(produto_mais_comprado, "id_cliente", "left")
    .withColumn(
        "segmento_cliente",
        F.when(F.col("total_compras") == 1, "Novo")
         .when((F.datediff(F.lit("2026-04-28"), F.col("data_ultima_compra")) > 180) | (F.col("total_compras").isNull()), "Inativo")
         .when(
             (F.col("total_compras") > 5) | (F.col("receita_total_cliente") > 1000),
             "Premium"
         )
         .otherwise("Recorrente")
    )
)

In [0]:
v_cliente_360 = v_cliente_360.fillna(0, subset=[
    "total_avaliacoes",
    "total_compras",
    "media_nota_produto",
    "media_nota_nps",
    "total_tickets",
    "tickets_abertos",
    "tickets_fechados",
    "ticket_medio",
    "receita_total_cliente",
    "total_sessoes",
    "total_produtos_visitados",
    "tempo_medio_sessao_seg"
])

v_cliente_360 = v_cliente_360.drop("timestamp_ingestion")

In [0]:
(
    v_cliente_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.gold.v_cliente_360")
)

print("rocket.gold.v_cliente_360 atualizada com sucesso!")
display(spark.read.table("rocket.gold.v_cliente_360"))

# Desempenho_produto


In [0]:
# Métricas de pedidos por produto 
#Calcula, por produto: total de pedidos realizados, unidades vendidas, receita total gerada e receita média por pedido.

metricas_pedidos_produto = (
    fat_pedidos
    .groupBy("id_produto")
    .agg(
        F.count("id_pedido").alias("total_pedidos"),
        F.sum("quantidade").alias("unidades_vendidas"),
        F.sum("valor_pedido").cast("decimal(15,2)").alias("receita_total"),
        F.round(F.avg("valor_pedido"), 2).cast("decimal(15,2)").alias("receita_media_por_pedido"),
    )
)

In [0]:
# Métricas de avaliações por produto 
#Calcula a média da nota do produto, a média do NPS e o percentual de clientes que recomendariam o produto. Produtos sem avaliações aparecem com nulo nessas colunas.

metricas_avaliacoes_produto = (
    fat_avaliacoes
    .groupBy("id_produto")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_produto"), 2).alias("media_nota_produto"),
        F.round(F.avg("nota_nps"), 2).alias("media_nota_nps"),
        F.round(
            F.sum(F.when(F.col("recomenda") == "sim", 1).otherwise(0)) /
            F.count("*") * 100, 2
        ).alias("pct_recomenda")
    )
)

In [0]:
# Métricas de tickets por produto 
#Conta o total de tickets associados a cada produto, cruzando pedidos com tickets via id_cliente. Permite identificar produtos que geram volume desproporcional de reclamações em relação às vendas.

metricas_tickets_produto = (
    fat_tickets
    .join(fat_pedidos.select("id_pedido", "id_produto"), "id_pedido", "left")
    .groupBy("id_produto")
    .agg(F.count("*").alias("total_tickets"))
)

In [0]:
# Métricas de clickstream por produto 
#Conta o total de eventos de navegação associados a cada produto, considerando apenas sessões de clientes identificados. Permite identificar os produtos mais visitados independentemente das vendas.
metricas_click_produto = (
    dim_clickstream
    .filter(F.col("flag_anonimo") == False)
    .filter(F.col("id_produto").isNotNull())
    .groupBy("id_produto")
    .agg(
        F.count("*").alias("total_visualizacoes"),
    )
)

In [0]:
#Une todas as métricas ao catálogo de produtos via left join. Adiciona a flag_alto_ticket, que marca produtos cuja taxa de tickets por pedido supera 0.22 — threshold definido com base na distribuição real dos dados, onde a taxa máxima observada foi 0.24 e a média ficou em torno de 0.20.

desempenho_produtos = (
    dim_catalogo
    .join(metricas_pedidos_produto,    "id_produto", "left")
    .join(metricas_avaliacoes_produto, "id_produto", "left")
    .join(metricas_tickets_produto,    "id_produto", "left")
    .join(metricas_click_produto,      "id_produto", "left")
    .withColumn(
        "flag_alto_ticket",
        F.when(
            F.col("total_tickets") / F.col("total_pedidos") > 0.22, True
        ).otherwise(False)
    )
)

desempenho_produtos = desempenho_produtos.drop("timestamp_ingestion", "timestamp_silver")

In [0]:
# Escrita na Gold 

(
    desempenho_produtos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.gold.desempenho_produtos")
)

print("rocket.gold.desempenho_produtos atualizada com sucesso!")
display(spark.read.table("rocket.gold.desempenho_produtos"))

# gold.kpis_vendas

### Vendas por categoria

In [0]:
df_fat_pedidos = spark.read.table('rocket.silver.fat_pedidos')
df_dim_clientes = spark.read.table('rocket.silver.dim_clientes')
df_dim_catalogo_produtos = spark.read.table('rocket.silver.dim_catalogo_produtos')

df_gold_kpi_vendas = (
    df_fat_pedidos
    .join(
        df_dim_clientes,
        on="id_cliente",
        how="inner"
    )
    .join(
        df_dim_catalogo_produtos,
        on="id_produto",
        how="inner"
    )
)

df_gold_kpi_vendas = (
    df_gold_kpi_vendas
    .withColumn('ano_venda', year(col('data_pedido')))
    .withColumn('mes_venda', month(col('data_pedido')))
)

df_gold_kpi_vendas_categoria = (
    df_gold_kpi_vendas
    .groupBy(
        'ano_venda',
        'mes_venda',
        'categoria'
    )
    .agg(
        round(
            sum(col("valor_pedido")),
            2
        ).alias("receita_total"),

        round(
            avg(col('valor_pedido')),
            2
        ).alias('ticket_medio'),

        countDistinct('id_pedido').alias('total_pedidos'),

        countDistinct('id_cliente').alias('total_clientes_unicos')
    )
)

df_gold_kpi_vendas_categoria = df_gold_kpi_vendas_categoria.dropna()

(
    df_gold_kpi_vendas_categoria
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.gold.kpi_por_categoria")
)


In [0]:
df_gold_kpi_vendas_categoria.display()

### Vendas por estado

In [0]:
df_fat_pedidos = spark.read.table('rocket.silver.fat_pedidos')
df_dim_clientes = spark.read.table('rocket.silver.dim_clientes')

df_gold_kpi_vendas = (
    df_fat_pedidos
    .join(
        df_dim_clientes,
        on="id_cliente",
        how="inner"
    )
)

df_gold_kpi_vendas = (
    df_gold_kpi_vendas
    .withColumn('ano_venda', year(col('data_pedido')))
    .withColumn('mes_venda', month(col('data_pedido')))
)

df_gold_kpi_vendas_estado = (
    df_gold_kpi_vendas
    .groupBy(
        'ano_venda',
        'mes_venda',
        'estado'
    )
    .agg(
        round(
            sum(col("valor_pedido")),
            2
        ).alias("receita_total"),

        round(
            avg(col('valor_pedido')),
            2
        ).alias('ticket_medio'),

        countDistinct('id_pedido').alias('total_pedidos'),

        countDistinct('id_cliente').alias('total_clientes_unicos')
    )
)

df_gold_kpi_vendas_estado = df_gold_kpi_vendas_estado.dropna()

(
    df_gold_kpi_vendas_estado
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.gold.kpi_por_estado")
)

In [0]:
df_gold_kpi_vendas_estado.display()

### Vendas por status

In [0]:
df_fat_pedidos = spark.read.table('rocket.silver.fat_pedidos')
df_dim_clientes = spark.read.table('rocket.silver.dim_clientes')

df_gold_kpi_vendas = (
    df_fat_pedidos
    .join(
        df_dim_clientes,
        on="id_cliente",
        how="inner"
    )
)

df_gold_kpi_vendas = (
    df_gold_kpi_vendas
    .withColumn('ano_venda', year(col('data_pedido')))
    .withColumn('mes_venda', month(col('data_pedido')))
)

df_gold_kpi_vendas_status = (
    df_gold_kpi_vendas
    .groupBy(
        'ano_venda',
        'mes_venda',
        'status'
    )
    .agg(
        round(
            sum(col("valor_pedido")),
            2
        ).alias("receita_total"),

        round(
            avg(col('valor_pedido')),
            2
        ).alias('ticket_medio'),

        countDistinct('id_pedido').alias('total_pedidos'),

        countDistinct('id_cliente').alias('total_clientes_unicos')
    )
)

df_gold_kpi_vendas_status = df_gold_kpi_vendas_status.dropna()

(
    df_gold_kpi_vendas_status
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("rocket.gold.kpi_por_status")
)

In [0]:
df_gold_kpi_vendas_status.display()

# gold.comportamento_digital

- id_cliente: Coluna para identificar o cliente
- total_sessoes: A quantidade de sessões que esse cliente tem no sistema, calculado contando a quantidade de vezes que o cliente tem uma sessão na tabela
- total_eventos: A quantidade de eventos que esse cliente já passou por, calculado contando a quantidade de vezes que o cliente tem um evento na tabela
- total_visualizacoes_produto: Quantas vezes o cliente entrou em uma página de produto
- total_compras_click: A quantidade total de vezes que o cliente comprou um produto, calculado contando a quantidade de vezes que um cliente teve um tipo_evento 'purchase' associado a ele
- taxa_conversao_click: A taxa, em porcentagem, entre compras e visualizações de um cliente, calculado usando o resultado do cálculo dos totais de purchase/product_view de um cliente
- taxa_abandono_carrinho: A taxa, em porcentagem, que o cliente abandona uma compra que ele ia fazer, calculado usando o resultado do cálculo de totais de abandon_cart/add_to_cart
- canal_predominante: O canal que o cliente mais usa, calculado verificando o canal mais predominante associado a um cliente
- produto_mais_visitado: O produto que o cliente mais visitou a página, calculado verificando o produto mais predominante nos 'product_view' de um cliente

In [0]:
window_canal = Window.partitionBy("id_cliente").orderBy(F.desc("contagem_canal"))
canal_predominante = (
    dim_clickstream
    .groupBy("id_cliente", "canal")
    .agg(F.count("*").alias("contagem_canal"))
    .withColumn("rank", F.row_number().over(window_canal))
    .filter(F.col("rank") == 1)
    .select("id_cliente", F.col("canal").alias("canal_predominante"))
)

window_produto = Window.partitionBy("id_cliente").orderBy(F.desc("contagem_produto"))
produto_mais_visitado = (
    dim_clickstream
    .filter(F.col("tipo_evento") == "product_view")
    .filter(F.col("id_produto").isNotNull())
    .groupBy("id_cliente", "id_produto")
    .agg(F.count("*").alias("contagem_produto"))
    .withColumn("rank", F.row_number().over(window_produto))
    .filter(F.col("rank") == 1)
    .join(dim_catalogo.select("id_produto", "nome_produto"), "id_produto", "left")
    .select("id_cliente", F.col("nome_produto").alias("produto_mais_visitado"))
)

df_gold_comportamento_digital = (
    dim_clickstream
    .groupBy("id_cliente")
    .agg(
        F.countDistinct("id_sessao").alias("total_sessoes"),
        F.count("*").alias("total_eventos"),
        F.sum(F.when(F.col("tipo_evento") == "product_view", 1).otherwise(0)).alias("total_visualizacoes_produto"),
        F.sum(F.when(F.col("tipo_evento") == "purchase", 1).otherwise(0)).alias("total_compras_click"),
        F.sum(F.when(F.col("tipo_evento") == "add_to_cart", 1).otherwise(0)).alias("total_add_to_cart"),
        F.sum(F.when(F.col("tipo_evento") == "abandon_cart", 1).otherwise(0)).alias("total_abandon_cart")
    )
    .withColumn(
        "taxa_conversao_click",
        F.when(F.col("total_visualizacoes_produto") > 0,
               F.round(F.col("total_compras_click") / F.col("total_visualizacoes_produto"), 4)
        ).otherwise(0)
    )
    .withColumn(
        "taxa_abandono_carrinho",
        F.when(F.col("total_add_to_cart") > 0,
               F.round(F.col("total_abandon_cart") / F.col("total_add_to_cart"), 4)
        ).otherwise(0)
    )
    .join(canal_predominante, "id_cliente", "left")
    .join(produto_mais_visitado, "id_cliente", "left")
    .select(
        "id_cliente",
        "total_sessoes",
        "total_eventos",
        "total_visualizacoes_produto",
        "total_compras_click",
        "taxa_conversao_click",
        "taxa_abandono_carrinho",
        "canal_predominante",
        F.when(F.col("produto_mais_visitado").isNull(), "Produto Indisponível").otherwise(F.col("produto_mais_visitado")).alias("produto_mais_visitado"),
    )
)

df_gold_comportamento_digital.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("rocket.gold.comportamento_digital")

In [0]:
df_gold_comportamento_digital.display()

# gold.analise_tickets

In [0]:
from pyspark.sql import functions as F

# Lendo as tabelas da camada Silver
df_tickets  = spark.table("rocket.silver.fat_suporte_tickets")
df_clientes = spark.table("rocket.silver.dim_clientes")
df_pedidos  = spark.table("rocket.silver.fat_pedidos")
df_produtos = spark.table("rocket.silver.dim_catalogo_produtos")

# Criando as Métricas do Cliente (LTV - Lifetime Value e Frequência)
# Agrupamos os pedidos por cliente para saber o total gasto e a quantidade de pedidos na vida dele
df_metricas_cliente = df_pedidos.groupBy("id_cliente").agg(
    F.count("id_pedido").alias("total_pedidos_cliente"),
    F.round(F.sum("valor_pedido"), 2).alias("receita_total_cliente")
)

# Cruzamento (JOINs)
df_gold_tickets = (
    df_tickets.alias("t")
    
    # Trazendo os dados do cliente
    .join(df_clientes.alias("c"), on="id_cliente", how="left")
    
    # Trazendo os dados do pedido daquele ticket
    .join(df_pedidos.alias("p"), on="id_pedido", how="left")
    
    # Trazendo os dados do produto (a chave de ligação aqui é com a fat_pedidos)
    .join(df_produtos.alias("pr"), on="id_produto", how="left")
    
    # Trazendo as métricas de vida do cliente que acabamos de calcular
    .join(df_metricas_cliente.alias("mc"), on="id_cliente", how="left")
)

# Selecionando, Renomeando e Formatando as colunas finais 
df_final = df_gold_tickets.select(
    # Dados do Ticket
    F.col("t.ticket_id").alias("id_ticket"),
    F.col("t.id_cliente"),
    F.concat_ws(" ", F.col("c.nome"), F.col("c.sobrenome")).alias("nome_cliente"), # Juntando Nome e Sobrenome
    F.col("t.tipo_problema"),
    F.col("t.status_ticket"),
    F.col("t.data_abertura"),
    F.col("t.data_resolucao"),
    F.col("t.tempo_resolucao_horas"),
    F.col("t.agente_suporte"),
    
    # Contexto de Produto e Pedido
    F.col("pr.nome_produto"),
    F.col("pr.categoria").alias("categoria_produto"), # Renomeando pra ficar igual ao doc
    F.col("p.valor_pedido"),
    
    # Métricas do Cliente
    F.col("mc.total_pedidos_cliente"),
    F.col("mc.receita_total_cliente")
)

df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("rocket.gold.analise_tickets")

# Visualizando a Tabela Gold Pronta
display(df_final)

# pedidos_por_cliente


In [ ]:
from pyspark.sql.functions import col, concat_ws, round as spark_round, current_timestamp

df_clientes = spark.read.table("rocket.silver.dim_clientes")
df_pedidos  = spark.read.table("rocket.silver.fat_pedidos")

# garante que só entram clientes com pedidos
df_gold = df_pedidos.alias("p").join(
    df_clientes.alias("c"),
    on="id_cliente",
    how="inner"
)

df_gold = df_gold.select(
    col("p.id_pedido"),
    col("c.id_cliente"),

    concat_ws(" ", col("c.nome"), col("c.sobrenome")).alias("nome_completo"),
    col("c.email"),
    col("c.cidade"),
    col("c.estado"),
    col("c.origem"),

    col("p.id_produto"),
    col("p.data_pedido"),
    col("p.valor_pedido"),
    col("p.quantidade"),
    col("p.status"),
    col("p.metodo_pagamento"),
)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("rocket.gold.pedidos_por_cliente")

display(df_gold)

#exportando tabelas para o .db

In [0]:
import sqlite3
from decimal import Decimal

# lista das tabelas gold
tabelas = [
    "rocket.gold.analise_tickets",
    "rocket.gold.comportamento_digital",
    "rocket.gold.desempenho_produtos",
    "rocket.gold.kpi_por_categoria",
    "rocket.gold.kpi_por_estado",
    "rocket.gold.kpi_por_status",
    "rocket.gold.v_cliente_360"
    "rocket.gold.pedidos_por_cliente"
]

# caminho do arquivo .db
db_path = "/tmp/gold.db"

# cria conexão SQLite
conn = sqlite3.connect(db_path)

for tabela in tabelas:
    df = spark.table(tabela)
    
    # cuidado: isso traz tudo pra memória
    pdf = df.toPandas()

    for col in pdf.columns:
        pdf[col] = pdf[col].map(lambda x: float(x) if isinstance(x, Decimal) else x)
    
    # nome da tabela no sqlite
    nome_tabela = tabela.split(".")[-1]
    
    pdf.to_sql(nome_tabela, conn, if_exists="replace", index=False)

conn.close()

In [0]:
import shutil

shutil.copy("/tmp/gold.db", "/Workspace/Users/joaopedrobarbosamarins@gmail.com/V-Commerce-CRM-360.db")